In [1]:
import pandas as pd
import numpy as np
import os
import gc
import joblib
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap
from sklearn.metrics import f1_score

# 1. Configuración de Rutas y Variables
dir_entrada = "../../Datos/Datos procesados"
dir_resultados = "../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia"
os.makedirs(dir_resultados, exist_ok=True)

columnas_a_cargar = [
    'CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER',
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 
    'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS'
]

vars_para_ohe = [
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CATEGORIA_CANCER' 
]

# AÑOS A INCLUIR (EXCLUYENDO 2020 y 2021)
años_sin_pandemia = [2019, 2022, 2023, 2024]

print("="*70)
print("INICIANDO ANÁLISIS DE SENSIBILIDAD (SIN PANDEMIA 2020-2021)")
print("="*70)

# 2. Cargar y Procesar Datos
df_lista = []
for año in años_sin_pandemia:
    ruta = os.path.join(dir_entrada, f"GRD_PROCESADO_{año}_DERIVADAS.csv")
    if os.path.exists(ruta):
        print(f"Cargando {año}...")
        df_temp = pd.read_csv(ruta, usecols=columnas_a_cargar, low_memory=False)
        df_temp['CATEGORIA_CANCER'] = df_temp['CATEGORIA_CANCER'].astype(str).str.split(':').str[0].str.replace('-', '_').str.strip()
        df_lista.append(df_temp)

df_crudo = pd.concat(df_lista, ignore_index=True)
del df_lista; gc.collect()

print("Aplicando One-Hot Encoding...")
df_ohe = pd.get_dummies(df_crudo, columns=vars_para_ohe, drop_first=True)
df_ohe.columns = df_ohe.columns.str.replace(' ', '_').str.replace('-', '_').str.upper()
del df_crudo; gc.collect()

# Para optimizar memoria
for col in df_ohe.select_dtypes(include=['float64']).columns:
    df_ohe[col] = df_ohe[col].astype('float32')
for col in df_ohe.select_dtypes(include=['int64']).columns:
    df_ohe[col] = df_ohe[col].astype('int32')

print("Separando cohorte oncológica y control...")
columnas_cancer = [col for col in df_ohe.columns if col.startswith('CATEGORIA_CANCER_') and 'SIN_CANCER' not in col]
mask_onco = df_ohe[columnas_cancer].sum(axis=1) > 0

df_onco = df_ohe[mask_onco]
df_control = df_ohe[~mask_onco]
del df_ohe; gc.collect()

# 3. Función de Entrenamiento y Extracción SHAP
def entrenar_y_extraer_shap(target_name, es_rf=False):
    print(f"\n--- Modelando {target_name} (Sin Pandemia) ---")
    
    # Split
    df_onco_train, df_onco_test = train_test_split(df_onco, test_size=0.20, random_state=42, stratify=df_onco[target_name])
    
    n_onco = len(df_onco_train)
    df_control_train = df_control.sample(n=n_onco, random_state=42)
    
    df_train = pd.concat([df_onco_train, df_control_train], ignore_index=True)
    
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD']
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    # Entrenar modelo
    if es_rf:
        modelo = RandomForestClassifier(n_estimators=500, max_depth=35, min_samples_split=10, class_weight='balanced', n_jobs=-1, random_state=42)
    else:
        modelo = xgb.XGBClassifier(learning_rate=0.3, max_depth=10, tree_method='hist', n_jobs=-1, random_state=42)
        
    print("Entrenando modelo...")
    modelo.fit(X_train, y_train)
    
    X_test_onco = df_onco_test.drop(columns=cols_drop, errors='ignore').reindex(columns=X_train.columns, fill_value=0)
    y_pred = modelo.predict(X_test_onco)
    explainer = shap.TreeExplainer(modelo)
    
    X_sample = X_test_onco.sample(n=min(5000, len(X_test_onco)), random_state=42)
    print(f"F1-Score en prueba: {f1_score(df_onco_test[target_name], y_pred, average='macro' if not es_rf else 'binary'):.4f}")
    
    print("Calculando SHAP y normalizando a porcentajes...")
    
    if es_rf: # MORTALIDAD (Binario)
        shap_values = explainer.shap_values(X_sample, check_additivity=False, approximate=True)
        shap_mat = shap_values[1] if isinstance(shap_values, list) else np.array(shap_values)[:, :, 1]
        
        # Filtro de constantes
        varianzas = X_sample.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_sample.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        if cols_a_eliminar:
            idx_a_eliminar = [X_sample.columns.get_loc(col) for col in cols_a_eliminar]
            X_sample = X_sample.drop(columns=cols_a_eliminar)
            shap_mat = np.delete(shap_mat, idx_a_eliminar, axis=1)

        # Cálculo de porcentajes para Binario (Clase 1 y Total son equivalentes en magnitud absoluta)
        shap_abs = np.abs(shap_mat).mean(axis=0)
        shap_pct_total = (shap_abs / shap_abs.sum()) * 100
        
        df_imp = pd.DataFrame({
            'Variable': X_sample.columns, 
            'Impacto_Total_SinPandemia': shap_pct_total,
            'Impacto_Clase1_SinPandemia': shap_pct_total 
        })

    else: # SEVERIDAD Y CONSUMO (Multiclase XGBoost)
        shap_values = explainer(X_sample).values
        # Filtro de constantes
        varianzas = X_sample.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_sample.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        if cols_a_eliminar:
            idx_a_eliminar = [X_sample.columns.get_loc(col) for col in cols_a_eliminar]
            X_sample = X_sample.drop(columns=cols_a_eliminar)
            shap_values = np.delete(shap_values, idx_a_eliminar, axis=1)

        # Cálculo de porcentajes Multiclase
        shap_abs_classes = np.abs(shap_values).mean(axis=0) # Promedio absoluto por feature y por clase
        
        # 1. Porcentaje Total (suma de todas las clases)
        impacto_crudo_total = shap_abs_classes.sum(axis=1)
        shap_pct_total = (impacto_crudo_total / impacto_crudo_total.sum()) * 100
        
        df_imp = pd.DataFrame({
            'Variable': X_sample.columns, 
            'Impacto_Total_SinPandemia': shap_pct_total
        })
        
        # 2. Porcentaje Específico de la Clase de Alto Riesgo
        if target_name == 'SEVERIDAD': # Clase 3
            impacto_crudo_c3 = shap_abs_classes[:, 3]
            df_imp['Impacto_Clase3_SinPandemia'] = (impacto_crudo_c3 / impacto_crudo_c3.sum()) * 100
        elif target_name == 'CONSUMO_RECURSOS': # Clase 2
            impacto_crudo_c2 = shap_abs_classes[:, 2]
            df_imp['Impacto_Clase2_SinPandemia'] = (impacto_crudo_c2 / impacto_crudo_c2.sum()) * 100

    # Ordenar por Impacto Total y guardar
    df_imp = df_imp.sort_values(by='Impacto_Total_SinPandemia', ascending=False)
    
    ruta_csv = os.path.join(dir_resultados, f"SHAP_Porcentajes_SinPandemia_{target_name}.csv")
    df_imp.to_csv(ruta_csv, index=False)
    print(f"-> Archivo con porcentajes guardado en {ruta_csv}")
    
    del modelo, X_train, y_train, df_train; gc.collect()

# 4. Ejecución
entrenar_y_extraer_shap('MORTALIDAD', es_rf=True)
entrenar_y_extraer_shap('SEVERIDAD', es_rf=False)
entrenar_y_extraer_shap('CONSUMO_RECURSOS', es_rf=False)

print("\n=== PROCESO FINALIZADO ===")

INICIANDO ANÁLISIS DE SENSIBILIDAD (SIN PANDEMIA 2020-2021)
Cargando 2019...
Cargando 2022...
Cargando 2023...
Cargando 2024...
Aplicando One-Hot Encoding...
Separando cohorte oncológica y control...

--- Modelando MORTALIDAD (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.4205
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia/SHAP_Porcentajes_SinPandemia_MORTALIDAD.csv

--- Modelando SEVERIDAD (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.7751
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia/SHAP_Porcentajes_SinPandemia_SEVERIDAD.csv

--- Modelando CONSUMO_RECURSOS (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.7592
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilid

In [4]:
import pandas as pd
import os

# 1. Configuración de rutas
dir_orig_mort = "../../Resultados/Resultados (etapa 5)/SHAP_MORTALIDAD/Valores SHAP (oncologicos)"
dir_orig_sev = "../../Resultados/Resultados (etapa 5)/SHAP_SEVERIDAD/Valores SHAP (oncologicos)"
dir_orig_cons = "../../Resultados/Resultados (etapa 5)/SHAP_CONSUMO_RECURSOS/Valores SHAP (oncologicos)"
dir_sin_pandemia = "../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia"

print("="*60)
print("CALCULANDO DIFERENCIAS (SIN PANDEMIA - ORIGINAL)")
print("="*60)

# --- 1. MORTALIDAD ---
print("\nProcesando Mortalidad...")
df_mort_orig = pd.read_csv(os.path.join(dir_orig_mort, "SHAP_Valores_MORTALIDAD_ONCO_PORCENTAJES.csv"))
df_mort_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_MORTALIDAD.csv"))

# Calcular rankings REALES antes de unir
df_mort_orig['Ranking_Original'] = df_mort_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_mort_sp['Ranking_Sin_Pandemia'] = df_mort_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Unir y calcular diferencias
df_mort_merge = pd.merge(df_mort_orig, df_mort_sp, on='Variable', how='inner')
df_mort_merge['Delta_Impacto_Total'] = df_mort_merge['Impacto_Total_SinPandemia'] - df_mort_merge['Impacto_Total']
df_mort_merge['Delta_Clase1'] = df_mort_merge['Impacto_Clase1_SinPandemia'] - df_mort_merge['Clase_1']

# Filtrar solo las que eran Top 20 original y ordenar
df_mort_top20 = df_mort_merge[df_mort_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()
cols_mort = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_1', 'Impacto_Clase1_SinPandemia', 'Delta_Clase1']
df_mort_top20[cols_mort].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_MORTALIDAD.csv"), index=False)
print("-> Diferencias_Pandemia_MORTALIDAD.csv guardado.")

# --- 2. SEVERIDAD ---
print("\nProcesando Severidad...")
df_sev_orig = pd.read_csv(os.path.join(dir_orig_sev, "SHAP_Valores_SEVERIDAD_ONCO_PORCENTAJES.csv"))
df_sev_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_SEVERIDAD.csv"))

# Calcular rankings REALES antes de unir
df_sev_orig['Ranking_Original'] = df_sev_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_sev_sp['Ranking_Sin_Pandemia'] = df_sev_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Unir y calcular diferencias
df_sev_merge = pd.merge(df_sev_orig, df_sev_sp, on='Variable', how='inner')
df_sev_merge['Delta_Impacto_Total'] = df_sev_merge['Impacto_Total_SinPandemia'] - df_sev_merge['Impacto_Total']
df_sev_merge['Delta_Clase3'] = df_sev_merge['Impacto_Clase3_SinPandemia'] - df_sev_merge['Clase_3']

# Filtrar solo las que eran Top 20 original y ordenar
df_sev_top20 = df_sev_merge[df_sev_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()
cols_sev = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_3', 'Impacto_Clase3_SinPandemia', 'Delta_Clase3']
df_sev_top20[cols_sev].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_SEVERIDAD.csv"), index=False)
print("-> Diferencias_Pandemia_SEVERIDAD.csv guardado.")

# --- 3. CONSUMO DE RECURSOS ---
print("\nProcesando Consumo de Recursos...")
df_cons_orig = pd.read_csv(os.path.join(dir_orig_cons, "SHAP_Valores_CONSUMO_RECURSOS_ONCO_PORCENTAJES.csv"))
df_cons_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_CONSUMO_RECURSOS.csv"))

# Calcular rankings REALES antes de unir
df_cons_orig['Ranking_Original'] = df_cons_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_cons_sp['Ranking_Sin_Pandemia'] = df_cons_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Unir y calcular diferencias
df_cons_merge = pd.merge(df_cons_orig, df_cons_sp, on='Variable', how='inner')
df_cons_merge['Delta_Impacto_Total'] = df_cons_merge['Impacto_Total_SinPandemia'] - df_cons_merge['Impacto_Total']
df_cons_merge['Delta_Clase2'] = df_cons_merge['Impacto_Clase2_SinPandemia'] - df_cons_merge['Clase_2']

# Filtrar solo las que eran Top 20 original y ordenar
df_cons_top20 = df_cons_merge[df_cons_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()
cols_cons = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_2', 'Impacto_Clase2_SinPandemia', 'Delta_Clase2']
df_cons_top20[cols_cons].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_CONSUMO_RECURSOS.csv"), index=False)
print("-> Diferencias_Pandemia_CONSUMO_RECURSOS.csv guardado.")

print("\n=== CÁLCULO DE DIFERENCIAS FINALIZADO ===")

CALCULANDO DIFERENCIAS (SIN PANDEMIA - ORIGINAL)

Procesando Mortalidad...
-> Diferencias_Pandemia_MORTALIDAD.csv guardado.

Procesando Severidad...
-> Diferencias_Pandemia_SEVERIDAD.csv guardado.

Procesando Consumo de Recursos...
-> Diferencias_Pandemia_CONSUMO_RECURSOS.csv guardado.

=== CÁLCULO DE DIFERENCIAS FINALIZADO ===
